# Detección de fraude financiero: de las transacciones a la decisión de revisar

**Taller reproducible y explicable que corre solo con CPU, en Databricks o en Google Colab.**

Elaborado por: **Oliver Camarena Palma · A00843123**

Repositorio: https://github.com/OliCamarera/ti2006b-2.12-A00843123

Notebook original de la clase: https://github.com/tec-research/ti2006b/blob/main/activities/2.11-databricks-workshop/notebooks/financial_fraud_demo.ipynb (repositorio del curso: https://github.com/tec-research/ti2006b)

**Datos en Databricks:** subí el CSV a `/Volumes/workspace/ti2006b_2_12/fraud_data/creditcard.csv` y lo guardé como la tabla Delta `workspace.ti2006b_2_12.creditcard`. Los dos tienen 284,807 filas.

Notebook base proporcionado por el Dr. Servio Palacios, Ph.D.

La idea de este trabajo es poder auditar un dataset de transacciones etiquetadas, entender cómo decide un árbol de decisión, comparar clasificación supervisada contra detección de anomalías, escoger un umbral de decisión usando validación y, al final, interpretar un conjunto de prueba cronológico que no se tocó durante el desarrollo.

El flujo que sigo es: **cargar → auditar → separar por tiempo → explorar solo con entrenamiento → entrenar → comparar en validación → elegir umbral → evaluar una vez en prueba → puntuar y exportar**.

Cada celda de código trae su explicación y hay que correrlas en orden. No se necesita GPU, clúster de Spark, credenciales de Kaggle ni TensorFlow. Todo el modelado es pandas/scikit-learn en el driver de Databricks, así que no hay que verlo como una prueba de rendimiento distribuido con Spark.

**Dataset:** transacciones con tarjeta de crédito de ULB/Worldline. Los resultados no se pueden comparar directamente con los de PaySim que aparecen en ejemplos más viejos. Las referencias están al final.


## 1. Abrir el notebook y preparar el entorno

**Colab:** entra a https://colab.research.google.com/ → Archivo → Subir notebook → elige este `.ipynb` y conéctalo a un runtime de Python con CPU. El cargador baja el CSV solito. Si vas a trabajar sin internet, primero sube `creditcard.csv` desde el panel de Archivos.

**Databricks:** Workspace → escoge una carpeta → Import → File → selecciona este `.ipynb` (o el `.py` de Databricks que venía con el material) → conecta un cómputo de Python → Run all.

Se necesita Python 3.11 o más reciente, con pandas, NumPy, matplotlib, scikit-learn e IPython. El cómputo ML de Databricks es lo más cómodo, aunque un entorno serverless de Python compatible también funciona.

Para esta actividad subí el CSV a un volumen de Unity Catalog, anoté su ruta completa en `/Volumes` y puse `CSV_VOLUME_PATH` y el nombre completo de la tabla en `DELTA_TABLE_NAME` en la celda de configuración. El notebook escribe el CSV como tabla Delta (sin sobrescribir una que ya exista), compara el número de filas y luego trabaja sobre esa tabla.

Si no hay acceso a internet, sube el CSV a un volumen de Unity Catalog y pon en `LOCAL_DATA_PATH` la ruta `/Volumes/<catalog>/<schema>/<volume>/creditcard.csv`.

Si falta alguna librería, instálala en una **celda aparte**, reinicia Python si la plataforma lo pide y vuelve a correr todo desde arriba:

```python
%pip install pandas==2.2.3 numpy==2.3.5 scikit-learn==1.8.0 matplotlib==3.10.8
```

Esas son las versiones con las que se verificó el notebook. Con otras versiones compatibles los números pueden variar un poquito. Lo esencial usa APIs de scikit-learn que existen desde la 1.4.


In [ ]:
from pathlib import Path
import hashlib
import os
import json
import platform
import shutil
import sys
import tempfile
import time
import urllib.request

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay, f1_score, precision_recall_curve,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

SEED = 42
LOCAL_DATA_PATH = ''  # Respaldo solo para correr local; no aplica a la entrega en Databricks.
CSV_VOLUME_PATH = '/Volumes/workspace/ti2006b_2_12/fraud_data/creditcard.csv'
DELTA_TABLE_NAME = 'workspace.ti2006b_2_12.creditcard'
OUTPUT_DIR = Path(tempfile.gettempdir()) / 'fraud_demo_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_MLFLOW = True  # Registra el experimento terminado en el workspace de Databricks.
MLFLOW_EXPERIMENT = ''  # Databricks: pon una ruta de experimento donde tengas permiso de escritura.

versions = {
    'python': platform.python_version(), 'numpy': np.__version__,
    'pandas': pd.__version__, 'scikit_learn': sklearn.__version__,
    'matplotlib': matplotlib.__version__,
}
display(pd.Series(versions, name='versión').to_frame())
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.grid': False})


## 2. Entender el dataset antes de entrenar cualquier modelo

El benchmark original tiene **284,807 transacciones**, de las cuales **492 están marcadas como fraude**. Son compras de tarjetahabientes europeos durante dos días de septiembre de 2013 [1, 2].

| Campo | Qué significa | Cómo lo usamos |
|---|---|---|
| `Time` | Segundos transcurridos desde la primera transacción | Ordenar las transacciones y hacer los cortes cronológicos |
| `V1`–`V28` | Variables que ya vienen transformadas con PCA y anonimizadas por confidencialidad | Predictores numéricos; no sabemos qué significaban originalmente |
| `Amount` | Monto de la transacción | Explorarlo y crear `LogAmount = log(1 + Amount)` |
| `Class` | `0`: legítima; `1`: fraude | Etiqueta para entrenar y evaluar, nunca como predictor |

El PCA no lo ajustamos nosotros, y el archivo no documenta cómo se hizo. Por eso, aunque el corte cronológico controla nuestro propio flujo, no puede garantizar que esa representación PCA se haya construido sin información del futuro. Tampoco hay identificadores de tarjeta o de cliente, así que no se puede validar separando clientes. Y con solo dos días de historia no se puede afirmar nada sobre el desempeño actual en producción ni sobre equidad.

El cargador usa el espejo público del CSV que cita el tutorial oficial de TensorFlow [2] y revisa el SHA-256 antes de leerlo. Si subes una copia a mano, tiene que coincidir con la misma versión. Si algún día el espejo cambia, lo correcto es investigar de dónde salió el archivo, no apagar la validación.


In [ ]:
DATA_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'
EXPECTED_SHA256 = '76274b691b16a6c49d3f159c883398e03ccd6d1ee12d9d8ee38f4b4b98551a89'

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

ON_DATABRICKS = bool(os.environ.get('DATABRICKS_RUNTIME_VERSION'))
if ON_DATABRICKS:
    if not CSV_VOLUME_PATH or not DELTA_TABLE_NAME:
        raise ValueError('Configura CSV_VOLUME_PATH y DELTA_TABLE_NAME con rutas reales de Unity Catalog.')
    data_path = Path(CSV_VOLUME_PATH).expanduser()
    if not data_path.is_file():
        raise FileNotFoundError(f'No se encontró el CSV en el volumen: {data_path}')
    volume_sdf = (spark.read.option('header', True).option('inferSchema', True).csv(CSV_VOLUME_PATH))
    csv_row_count = volume_sdf.count()
    if not spark.catalog.tableExists(DELTA_TABLE_NAME):
        volume_sdf.write.format('delta').mode('overwrite').saveAsTable(DELTA_TABLE_NAME)
    delta_sdf = spark.table(DELTA_TABLE_NAME)
    delta_row_count = delta_sdf.count()
    if csv_row_count != delta_row_count:
        raise ValueError(f'No coincide el número de filas: CSV del volumen={csv_row_count:,}, tabla Delta={delta_row_count:,}.')
    print(f'Ruta del CSV: {CSV_VOLUME_PATH} ({csv_row_count:,} filas)')
    print(f'Tabla Delta: {DELTA_TABLE_NAME} ({delta_row_count:,} filas)')
    raw = delta_sdf.toPandas()
else:
    if LOCAL_DATA_PATH:
        data_path = Path(LOCAL_DATA_PATH).expanduser()
        if not data_path.is_file():
            raise FileNotFoundError(f'No se encontró el CSV: {data_path}')
    elif Path('creditcard.csv').is_file():
        data_path = Path('creditcard.csv')
    else:
        data_path = Path(tempfile.gettempdir()) / 'fraud_demo_data' / 'creditcard.csv'
        data_path.parent.mkdir(parents=True, exist_ok=True)
        if not data_path.is_file():
            partial_path = data_path.with_suffix('.download')
            try:
                request = urllib.request.Request(DATA_URL, headers={'User-Agent': 'fraud-teaching-demo/1.0'})
                with urllib.request.urlopen(request, timeout=120) as response:
                    with partial_path.open('wb') as handle:
                        shutil.copyfileobj(response, handle)
                partial_path.replace(data_path)
            except Exception as exc:
                partial_path.unlink(missing_ok=True)
                raise RuntimeError('Falló la descarga. Configura LOCAL_DATA_PATH o una ruta de volumen de Databricks accesible.') from exc
    raw = pd.read_csv(data_path)

dataset_sha256 = sha256_file(data_path)
if dataset_sha256 != EXPECTED_SHA256:
    raise ValueError('El checksum del dataset no coincide con la versión verificada. Revisa el archivo o la fuente.')
print(f'Se cargaron {len(raw):,} filas × {raw.shape[1]} columnas; SHA-256 verificado.')

## 3. Auditar etiquetas, valores faltantes, valores inválidos y duplicados

`isna()` cuenta los faltantes, pero no considera faltante un texto con puros espacios. Aquí exigimos que todas las columnas sean numéricas y, si algo viene mal, el notebook truena a propósito en vez de convertirlo en silencio o rellenarlo con cero. Un monto de cero es un valor que sí puede pasar; no es lo mismo que un monto faltante.

Las filas duplicadas exactas pueden hacer que la misma observación aparezca en entrenamiento y en evaluación. Como el benchmark no trae ID de transacción, también podría tratarse de transacciones reales que simplemente se ven idénticas.

**Seguimos la convención de la clase: nos quedamos con la primera fila de cada duplicado exacto, y lo dejamos documentado.**

No quitamos filas solo porque el monto sea muy alto o los valores de PCA se vean raros, porque justo esos podrían ser los fraudes que buscamos. Los conteos de la auditoría usan el archivo completo únicamente para revisar integridad; la exploración y la selección de modelos de más abajo usan solo entrenamiento y validación.


In [ ]:
expected_columns = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount', 'Class']
assert list(raw.columns) == expected_columns, 'El esquema o el orden de columnas no es el esperado.'
assert raw.shape == (284807, 31), 'Las dimensiones del dataset no son las esperadas.'
assert raw['Class'].value_counts().to_dict() == {0: 284315, 1: 492}
assert all(pd.api.types.is_numeric_dtype(raw[c]) for c in expected_columns)
assert not raw.isna().any().any(), 'Hay datos faltantes; revísalos antes de modelar.'
assert np.isfinite(raw.to_numpy(dtype=float)).all(), 'Se encontró un valor numérico no finito.'
assert raw['Class'].isin([0, 1]).all()
assert raw['Amount'].ge(0).all() and raw['Time'].ge(0).all()

duplicates = int(raw.duplicated().sum())
data = raw.drop_duplicates(keep='first').sort_values('Time', kind='stable').reset_index(drop=True)
audit = pd.DataFrame({
    'stage': ['original', 'after_exact_deduplication'],
    'rows': [len(raw), len(data)],
    'frauds': [int(raw['Class'].sum()), int(data['Class'].sum())],
    'missing_cells': [int(raw.isna().sum().sum()), int(data.isna().sum().sum())],
})
audit['fraud_rate'] = audit['frauds'] / audit['rows']
display(audit)
print(f'Filas duplicadas exactas eliminadas: {duplicates:,}')


## 4. Separar por tiempo antes de desarrollar el modelo

**Entrenamiento** se queda más o menos con el **60%** más antiguo, **validación** con el siguiente **20%** y **prueba** con el **20%** más reciente. Las transacciones con el mismo timestamp siempre quedan del mismo lado del corte. Así respetamos el orden en el tiempo y dejamos que la proporción de fraudes cambie de forma natural entre conjuntos. Una división aleatoria estratificada sería un buen ejercicio IID, pero no simula lo que pasa en la vida real: entrenar hoy y puntuar transacciones que todavía no han ocurrido.

**Entrenamiento:** ajustar los modelos y el escalador.
**Validación:** comparar modelos y escoger el umbral.
**Prueba:** evaluar una sola vez la decisión ya congelada.

No usamos oversampling, undersampling ni SMOTE. El peso por clase cambia la función de pérdida durante el entrenamiento, pero no altera la proporción real de fraudes en la evaluación. En un sistema real, además, las etiquetas tendrían que estar disponibles antes de la fecha de corte del entrenamiento.


In [ ]:
train_cut, validation_cut = data['Time'].quantile([0.60, 0.80]).to_numpy()
train = data.loc[data['Time'] <= train_cut].copy()
validation = data.loc[(data['Time'] > train_cut) & (data['Time'] <= validation_cut)].copy()
test = data.loc[data['Time'] > validation_cut].copy()

split_summary = pd.DataFrame([
    {'split': name, 'rows': len(part), 'frauds': int(part['Class'].sum()),
     'fraud_rate': part['Class'].mean(), 'first_second': part['Time'].min(),
     'last_second': part['Time'].max()}
    for name, part in [('train', train), ('validation', validation), ('test', test)]
])
assert len(train) + len(validation) + len(test) == len(data)
assert train['Time'].max() < validation['Time'].min()
assert validation['Time'].max() < test['Time'].min()
assert all(part['Class'].nunique() == 2 for part in [train, validation, test])
display(split_summary)


## 5. Explorar el entrenamiento y caer (a propósito) en la trampa de la exactitud

Solo usamos los datos de entrenamiento para ver cómo se distribuyen los montos. `log1p` comprime la cola larga hacia la derecha y no tiene problema con los ceros. Los dos histogramas están normalizados por separado, así que la altura muestra la forma de la distribución, no cuántos fraudes hay. Y que una gráfica de dos variables se vea separable no quiere decir que el problema completo lo sea.

**Antes de correr la celda:** si solo alrededor del 0.2% de las transacciones son fraude, ¿qué exactitud saca un modelo que siempre dice "legítima"? ¿Ese modelo detiene algún fraude?


In [ ]:
display(train.groupby('Class')['Amount'].describe())
fig, axes = plt.subplots(1, 3, figsize=(14, 3.7))
counts = train['Class'].value_counts().reindex([0, 1])
axes[0].bar(['Legítima', 'Fraude'], counts, color=['#2563eb', '#dc2626'])
axes[0].set_yscale('log')
axes[0].set_ylabel('Transacciones de entrenamiento (escala log)')
axes[0].set_title('Desbalance de clases muy fuerte')
for label, color in [(0, '#2563eb'), (1, '#dc2626')]:
    values = np.log1p(train.loc[train['Class'] == label, 'Amount'])
    axes[1].hist(values, bins=35, density=True, alpha=0.5, color=color,
                 label='Fraude' if label else 'Legítima')
axes[1].set(xlabel='log(1 + Amount)', ylabel='Densidad', title='El monto por sí solo no detecta fraude')
axes[1].legend()
scatter = train.sample(n=min(10000, len(train)), random_state=SEED)
for label, color in [(0, '#2563eb'), (1, '#dc2626')]:
    points = scatter.loc[scatter['Class'] == label]
    axes[2].scatter(points['V14'], points['V17'], s=12, alpha=0.45,
                    color=color, label='Fraude' if label else 'Legítima')
axes[2].set(xlabel='V14 (PCA)', ylabel='V17 (PCA)', title='10,000 filas de entrenamiento muestreadas')
axes[2].legend()
fig.tight_layout()
plt.show()
print(f'Exactitud de "siempre legítima" en entrenamiento: {1 - train["Class"].mean():.4%}; recall de fraude: 0%.')


## 6. Preparar los predictores sin filtrar la etiqueta

Usamos `V1`–`V28` más `LogAmount`. Dejamos fuera `Class` y también `Time` como predictor, para que el modelo no se apoye en el origen arbitrario de los segundos transcurridos. `Time` sigue sirviendo para definir el orden de evaluación. `log1p` es una transformación fija, fila por fila, que no aprende nada de los datos. El escalador de la regresión logística se ajusta más adelante dentro de un pipeline, usando solo filas de entrenamiento.


In [ ]:
FEATURES = [f'V{i}' for i in range(1, 29)] + ['LogAmount']

def make_features(frame):
    features = frame[[f'V{i}' for i in range(1, 29)]].copy()
    features['LogAmount'] = np.log1p(frame['Amount'])
    assert list(features.columns) == FEATURES
    assert np.isfinite(features.to_numpy()).all()
    return features

X_train, y_train = make_features(train), train['Class'].astype(int)
X_validation, y_validation = make_features(validation), validation['Class'].astype(int)
assert 'Class' not in FEATURES and 'Time' not in FEATURES
print('Matriz de predictores:', X_train.shape)


## 7. ¿Qué modelos vamos a entrenar y por qué?

| Modelo | Tipo de aprendizaje | Para qué sirve y decisiones clave |
|---|---|---|
| Siempre legítima | Línea base dummy | Mostrar por qué la exactitud sola engaña |
| Regresión logística | Supervisado | Línea base lineal en log-odds; con escalado y pesos de clase balanceados |
| Árbol de decisión | Supervisado | Divisiones no lineales fáciles de interpretar; profundidad 4 y mínimo 20 filas por hoja |
| Random forest | Supervisado | Promedia 100 árboles aleatorizados; profundidad 12 y mínimo 5 filas por hoja |
| Isolation Forest | Ajuste no supervisado | Encuentra observaciones fáciles de aislar; 100 árboles y 256 filas muestreadas por árbol |

Un árbol de decisión escoge la variable y el umbral que más reducen la impureza ponderada de las clases. Si las proporciones de clase son p₀ y p₁, **Gini = 1 − p₀² − p₁²**, y un nodo puro tiene Gini igual a cero. `max_depth` limita cuántas decisiones seguidas puede tomar el árbol y `min_samples_leaf` evita hojas demasiado chiquitas. Con `class_weight='balanced'`, cada ejemplo recibe un peso inversamente proporcional a la frecuencia de su clase, por lo que los puntajes de las hojas no deben leerse como probabilidades reales y calibradas de fraude. El bosque usa `balanced_subsample`, que recalcula esos pesos dentro de cada muestra bootstrap. En general, un bosque es más difícil de explicar que un solo árbol pequeño.

Isolation Forest se ajusta con **todo X_train sin usar y_train**. Le cambiamos el signo a `score_samples` porque en el puntaje original, más bajo significa más anómalo. Ese riesgo no es una probabilidad. Después sí lo evaluamos con etiquetas, y en cuanto escogemos su umbral usando validación etiquetada, esa decisión de despliegue ya usa información de las etiquetas [3–6]. Una anomalía no es automáticamente un fraude. Los hiperparámetros de abajo son valores fijos para la clase, no el resultado de una búsqueda exhaustiva.


In [ ]:
models = {
    'Always legitimate': DummyClassifier(strategy='constant', constant=0),
    'Logistic regression': Pipeline([
        ('scale', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1500,
                                     solver='lbfgs', random_state=SEED)),
    ]),
    'Decision tree': DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=20, class_weight='balanced', random_state=SEED),
    'Random forest': RandomForestClassifier(
        n_estimators=100, max_depth=12, min_samples_leaf=5,
        class_weight='balanced_subsample', n_jobs=2, random_state=SEED),
    'Isolation Forest': IsolationForest(
        n_estimators=100, max_samples=256, contamination='auto',
        n_jobs=2, random_state=SEED),
}

fit_seconds = {}
for name, model in models.items():
    started = time.perf_counter()
    if name == 'Isolation Forest':
        model.fit(X_train)  # Aquí no se le pasan etiquetas.
    else:
        model.fit(X_train, y_train)
    fit_seconds[name] = time.perf_counter() - started
display(pd.Series(fit_seconds, name='fit_seconds').to_frame().round(2))


## 8. Usar métricas que sí respondan la pregunta de fraude

**Precisión = TP/(TP+FP):** de las alertas, ¿cuántas eran fraude de verdad?

**Recall = TP/(TP+FN):** de los fraudes, ¿cuántos detectamos?

**F1:** media armónica de precisión y recall para un umbral dado.

**Average precision (AP):** resume qué tan bien ordena el modelo en la curva precisión–recall, considerando todos los umbrales [7]. No es exactamente lo mismo que el PR-AUC calculado con trapecios.

**ROC-AUC:** qué tan bien separa en el ranking los fraudes de las legítimas. Sirve como complemento del AP, pero no dice nada sobre la carga de trabajo de las alertas.

**Tasa de alertas:** fracción de transacciones que se mandan a revisión. Un puntaje constante tiene un AP igual a la prevalencia de positivos y un ROC-AUC de 0.5 (cuando existen ambas clases).

Primero comparamos los rankings en validación. Después reportamos las métricas que dependen del umbral usando `0.5` para los modelos supervisados y la frontera de anomalía por defecto de Isolation Forest. Esos valores por defecto son un punto de partida, no una política optimizada. Cuando no hay predicciones positivas reportamos precisión cero. Todos los conteos y métricas están sin ponderar y sobre la distribución natural de los datos.


In [ ]:
def risk_scores(name, model, features):
    if name == 'Isolation Forest':
        return -model.score_samples(features)
    fraud_column = int(np.flatnonzero(model.classes_ == 1)[0])
    return model.predict_proba(features)[:, fraud_column]

def operating_metrics(labels, scores, threshold):
    predictions = (np.asarray(scores) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    return {
        'threshold': float(threshold), 'AP': average_precision_score(labels, scores),
        'ROC_AUC': roc_auc_score(labels, scores),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'F1': f1_score(labels, predictions, zero_division=0),
        'accuracy': accuracy_score(labels, predictions), 'alert_rate': predictions.mean(),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
    }

validation_scores = {}
default_thresholds = {}
validation_rows = []
for name, model in models.items():
    scores = risk_scores(name, model, X_validation)
    validation_scores[name] = scores
    threshold = -float(model.offset_) if name == 'Isolation Forest' else 0.5
    default_thresholds[name] = threshold
    validation_rows.append({'model': name, **operating_metrics(y_validation, scores, threshold)})
validation_metrics = pd.DataFrame(validation_rows).set_index('model')
display(validation_metrics.sort_values('AP', ascending=False).round(5))

fig, ax = plt.subplots(figsize=(8, 4.5))
for name, scores in validation_scores.items():
    precision, recall, _ = precision_recall_curve(y_validation, scores)
    ax.step(recall, precision, where='post',
            label=f'{name}: AP={average_precision_score(y_validation, scores):.3f}')
ax.axhline(y_validation.mean(), color='gray', linestyle='--', label='Prevalencia en validación')
ax.set(xlabel='Recall', ylabel='Precisión', title='Curvas precisión–recall en validación',
       xlim=(0, 1), ylim=(0, 1.02))
ax.legend(loc='lower left', fontsize=8)
fig.tight_layout()
plt.show()


## 9. Abrir el árbol en vez de tratarlo como caja negra

El árbol entrenado tiene como máximo cuatro niveles de decisión. En el diagrama solo se ven los tres primeros para que se pueda leer; las reglas en texto muestran todos. Para seguir un camino: irse a la izquierda significa que la condición `variable <= umbral` se cumple. Cada hoja predice una clase con base en los conteos ponderados de entrenamiento. Como usamos pesos por clase, las proporciones y los valores de cada nodo reflejan esos pesos, mientras que `samples` sí cuenta filas reales de entrenamiento. Nombres como V14 vienen del PCA; no son categorías de comercio ni explicaciones financieras que una persona pueda leer directamente.

La importancia por impureza nos dice qué variables redujeron más la impureza en entrenamiento, pero no demuestra causalidad ni estabilidad. Si hay predictores correlacionados, pueden repartirse o sustituirse la importancia. Esta visualización explica el árbol aunque al final otro modelo gane la comparación en validación.


In [ ]:
tree_model = models['Decision tree']
fig, ax = plt.subplots(figsize=(17, 7))
plot_tree(tree_model, feature_names=FEATURES, class_names=['Legítima', 'Fraude'],
          filled=True, rounded=True, max_depth=2, fontsize=8, ax=ax)
ax.set_title('Árbol de decisión de profundidad 4: se muestran los tres primeros niveles')
fig.tight_layout()
plt.show()
print(export_text(tree_model, feature_names=FEATURES, decimals=3))
tree_importance = pd.Series(tree_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
display(tree_importance.head(10).rename('impurity_importance').to_frame())


## 10. Congelar un modelo y escoger el umbral de revisión solo con validación

La regla para escoger el modelo, declarada de antemano, es: **el AP de validación más alto entre los candidatos supervisados** (regresión logística, árbol de decisión y random forest). Isolation Forest está como comparación de detección de anomalías y el dummy como línea base. El ganador nunca se escoge viendo los resultados de prueba.

La política de operación también se declara antes: **al menos 80% de recall en validación y, entre los umbrales que lo cumplan, el de mayor precisión**. Si hay empate, gana el umbral más alto. Es una política para fines didácticos, no la meta contractual de un banco. `precision_recall_curve` regresa un punto extra de precisión/recall que no tiene umbral, así que lo quitamos. En puntajes continuos el umbral no tiene por qué ser 0.5. Como validación tiene pocos fraudes, el umbral elegido tiene bastante incertidumbre y el 80% de recall no está garantizado con datos futuros.

El modelo seleccionado se queda tal cual. Si lo reentrenáramos con entrenamiento + validación, cambiaría la distribución de sus puntajes y el umbral que escogimos para el modelo anterior ya no serviría [8].


In [ ]:
supervised_candidates = ['Logistic regression', 'Decision tree', 'Random forest']
selected_name = validation_metrics.loc[supervised_candidates, 'AP'].idxmax()
selected_model = models[selected_name]
TARGET_RECALL = 0.80

precision, recall, thresholds = precision_recall_curve(
    y_validation, validation_scores[selected_name])
threshold_table = pd.DataFrame({
    'threshold': thresholds, 'precision': precision[:-1], 'recall': recall[:-1],
})
feasible = threshold_table.loc[threshold_table['recall'] >= TARGET_RECALL]
if feasible.empty:
    raise RuntimeError('Ningún umbral de validación cumple la política de recall declarada.')
chosen = feasible.sort_values(['precision', 'threshold'], ascending=[False, False]).iloc[0]
selected_threshold = float(chosen['threshold'])
validation_selected = operating_metrics(
    y_validation, validation_scores[selected_name], selected_threshold)
assert validation_selected['recall'] >= TARGET_RECALL
print('Modelo seleccionado:', selected_name)
display(pd.Series(validation_selected, name='validation_at_frozen_threshold').to_frame())

fig, ax = plt.subplots(figsize=(8, 3.7))
ax.plot(threshold_table['threshold'], threshold_table['precision'], label='Precisión en validación')
ax.plot(threshold_table['threshold'], threshold_table['recall'], label='Recall en validación')
ax.axvline(selected_threshold, color='black', linestyle='--', label='Umbral congelado')
ax.axhline(TARGET_RECALL, color='gray', linestyle=':')
ax.set(xlabel='Umbral de riesgo', ylabel='Métrica', title=f'{selected_name}: compromiso de la política', ylim=(0, 1.03))
ax.legend()
fig.tight_layout()
plt.show()


## 11. Evaluar una sola vez la política congelada en la ventana de prueba

Esta es la primera vez que usamos las filas de prueba para predecir y evaluar. Comparamos el modelo seleccionado con su umbral por defecto contra el umbral elegido en validación, y además contra el dummy. La comparación es solo para explicar: **no se vale cambiar de umbral ni de modelo después de ver esta tabla**.

En la matriz de confusión, las filas son la clase real y las columnas la predicha. Los falsos negativos son fraudes que se nos fueron. Los falsos positivos son transacciones legítimas que alguien tendría que revisar. Un recall muy alto con demasiados falsos positivos puede rebasar la capacidad de un equipo de analistas.

Los intervalos de Wilson de abajo muestran la incertidumbre que viene de tener pocos fraudes y pocas alertas etiquetadas. Usan una aproximación binomial y no consideran la dependencia entre transacciones ni la incertidumbre de haber escogido modelo y umbral. El cambio en la distribución de los datos con el tiempo es otra fuente de error.


In [ ]:
X_test, y_test = make_features(test), test['Class'].astype(int)
test_scores = risk_scores(selected_name, selected_model, X_test)
test_results = operating_metrics(y_test, test_scores, selected_threshold)
test_default = operating_metrics(y_test, test_scores, default_thresholds[selected_name])
dummy_scores = risk_scores('Always legitimate', models['Always legitimate'], X_test)
test_baseline = operating_metrics(y_test, dummy_scores, 0.5)
test_comparison = pd.DataFrame([
    {'policy': 'Always legitimate', **test_baseline},
    {'policy': f'{selected_name}: default', **test_default},
    {'policy': f'{selected_name}: frozen validation policy', **test_results},
]).set_index('policy')
display(test_comparison.round(5))

test_predictions = (test_scores >= selected_threshold).astype(int)
fig, ax = plt.subplots(figsize=(5.5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, labels=[0, 1], display_labels=['Legítima', 'Fraude'],
    cmap='Blues', colorbar=False, values_format='d', ax=ax)
ax.set_title('Prueba (futuro): política congelada de validación')
fig.tight_layout()
plt.show()

def wilson_interval(successes, trials, z=1.96):
    if trials == 0:
        return (float('nan'), float('nan'))
    p = successes / trials
    denominator = 1 + z * z / trials
    center = (p + z * z / (2 * trials)) / denominator
    margin = z * np.sqrt(p * (1 - p) / trials + z * z / (4 * trials * trials)) / denominator
    return float(center - margin), float(center + margin)

recall_interval = wilson_interval(test_results['TP'], test_results['TP'] + test_results['FN'])
precision_interval = wilson_interval(test_results['TP'], test_results['TP'] + test_results['FP'])
display(pd.DataFrame([
    {'metric': 'recall', 'estimate': test_results['recall'], 'lower_95': recall_interval[0], 'upper_95': recall_interval[1]},
    {'metric': 'precision', 'estimate': test_results['precision'], 'lower_95': precision_interval[0], 'upper_95': precision_interval[1]},
]))


## 12. Traducir las predicciones a carga de trabajo y a un escenario de costos ilustrativo

Una alerta significa **mandar a revisión**, no bloquear la tarjeta ni declarar culpable a nadie. Reportamos alertas por cada 10,000 transacciones de prueba. La función de costo supone que cada fraude no detectado cuesta 100 unidades monetarias y cada alerta falsa cuesta 1. Son supuestos inventados para la clase, no montos sacados del dataset ni un ROI de producción. Deja fuera el costo de investigar los verdaderos positivos, lo que se recupera, la molestia para el cliente y que no todos los fraudes son del mismo monto. Se pueden mover estos supuestos para discutir la política, pero sin reajustar nada con base en los resultados de prueba.


In [ ]:
COST_MISSED_FRAUD = 100.0
COST_FALSE_ALERT = 1.0
cost_scenario = test_comparison[['FP', 'FN', 'TP', 'alert_rate']].copy()
cost_scenario['alerts_per_10000'] = 10000 * cost_scenario['alert_rate']
cost_scenario['illustrative_cost'] = (
    COST_MISSED_FRAUD * cost_scenario['FN'] + COST_FALSE_ALERT * cost_scenario['FP'])
display(cost_scenario.round(2))
print(f'La política congelada manda a revisión {test_predictions.sum():,} de {len(test):,} transacciones.')


## 13. Puntuar registros nuevos con las mismas transformaciones y la misma política

Esta función revisa que vengan las entradas numéricas necesarias, aplica la misma transformación `make_features` y regresa el riesgo junto con una bandera de revisión. No necesita `Class` ni `Time`. El puntaje es una estimación de riesgo del modelo, no una afirmación de que haya fraude. El ejemplo usa cinco transacciones de prueba solo para mostrar cómo se usa la interfaz; no es una estimación adicional de desempeño ni un modelo nuevo.


In [ ]:
def score_transactions(frame):
    required = [f'V{i}' for i in range(1, 29)] + ['Amount']
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f'Faltan columnas obligatorias: {missing}')
    if not all(pd.api.types.is_numeric_dtype(frame[c]) for c in required):
        raise ValueError('Las entradas obligatorias deben ser numéricas.')
    values = frame[required].to_numpy(dtype=float)
    if not np.isfinite(values).all() or frame['Amount'].lt(0).any():
        raise ValueError('Las entradas deben ser finitas y Amount no puede ser negativo.')
    scores = risk_scores(selected_name, selected_model, make_features(frame))
    return pd.DataFrame({
        'risk_score': scores, 'review_flag': (scores >= selected_threshold).astype(int),
    }, index=frame.index)

demo_input = test.head(5).drop(columns=['Class', 'Time'])
demo_scores = score_transactions(demo_input)
assert np.allclose(demo_scores['risk_score'], test_scores[:5])
display(demo_scores)


## 14. Exportar artefactos reproducibles de la evaluación

Guardamos la auditoría, los conteos de cada conjunto, la comparación en validación, la comparación en prueba con la política congelada, los puntajes de prueba fila por fila y un manifiesto con el hash del dataset, el entorno, las variables, los hiperparámetros y el umbral. `source_row_after_dedup` es una referencia local a la fila, no un ID real de transacción. Este benchmark no trae identificadores sensibles de tarjetas.

La carpeta de salida por defecto es temporal. En Colab los archivos se bajan desde el panel de Archivos; en Databricks conviene cambiar `OUTPUT_DIR` en la sección 1 por una carpeta `/Volumes/...` autorizada para que no se pierdan. El modelo se queda en memoria: los CSV/JSON exportados por sí solos no son un modelo desplegable.


In [ ]:
audit.to_csv(OUTPUT_DIR / 'data_audit.csv', index=False)
split_summary.to_csv(OUTPUT_DIR / 'split_summary.csv', index=False)
validation_metrics.to_csv(OUTPUT_DIR / 'validation_metrics.csv')
test_comparison.to_csv(OUTPUT_DIR / 'test_metrics.csv')
scored_test = pd.DataFrame({
    'source_row_after_dedup': test.index, 'Time': test['Time'].to_numpy(),
    'Amount': test['Amount'].to_numpy(), 'actual_class': y_test.to_numpy(),
    'risk_score': test_scores, 'review_flag': test_predictions,
})
scored_test.to_csv(OUTPUT_DIR / 'test_scores.csv', index=False)
manifest = {
    'dataset_url': DATA_URL, 'dataset_sha256': dataset_sha256,
    'original_rows': len(raw), 'deduplicated_rows': len(data),
    'duplicate_policy': 'keep first exact row', 'seed': SEED,
    'split': 'chronological quantiles 60/20/20; tied timestamps kept together',
    'train_cut_seconds': float(train_cut), 'validation_cut_seconds': float(validation_cut),
    'features': FEATURES, 'versions': versions, 'selected_model': selected_name,
    'selected_model_parameters': {
        key: value for key, value in selected_model.get_params().items()
        if value is None or isinstance(value, (str, bool, int, float))
    },
    'selection_metric': 'validation average precision, supervised candidates only',
    'target_validation_recall': TARGET_RECALL, 'selected_threshold': selected_threshold,
    'validation_metrics_at_selected_threshold': validation_selected,
    'test_metrics_at_selected_threshold': test_results,
    'test_recall_wilson_95': recall_interval,
    'test_precision_wilson_95': precision_interval,
    'cost_assumptions': {'missed_fraud': COST_MISSED_FRAUD, 'false_alert': COST_FALSE_ALERT},
}
with (OUTPUT_DIR / 'run_manifest.json').open('w') as handle:
    json.dump(manifest, handle, indent=2, allow_nan=False)
print('Artefactos:', OUTPUT_DIR)
print('\n'.join(sorted(path.name for path in OUTPUT_DIR.iterdir() if path.is_file())))


## 15. Opcional: registrar el experimento con MLflow

Esta sección no es parte del taller básico. Para usarla, instala MLflow si no está, pon `RUN_MLFLOW=True` y vuelve a correr esta parte después del flujo principal. En Databricks, pon en `MLFLOW_EXPERIMENT` una ruta de experimento de tu workspace donde tengas permiso de escritura; no hay ningún ID de experimento fijo en el código. En Colab, si no hay tracking URI configurado, se usa un directorio temporal local, así que no se crea un experimento ni una interfaz alojada en Databricks.

Lo que se registra es **el modelo ya ajustado que se seleccionó**, su umbral elegido en validación, la firma de entrada y las mismas métricas de prueba que calculamos arriba. El modelo sklearn registrado espera las 29 variables **ya transformadas**, no `Amount` en crudo; el manifiesto explica cómo construirlas. El umbral de revisión queda como parámetro del run y también tendría que aplicarlo quien sirva el modelo. Esto es seguimiento de experimentos, no un despliegue a producción. Esta integración necesita MLflow funcionando y credenciales/permisos si se usa un servidor remoto; la verificación local por defecto no la prueba.


In [ ]:
if RUN_MLFLOW:
    import os
    import mlflow
    import mlflow.sklearn
    from mlflow.models import infer_signature

    if not os.environ.get('DATABRICKS_RUNTIME_VERSION') and not os.environ.get('MLFLOW_TRACKING_URI'):
        mlflow.set_tracking_uri((Path(tempfile.gettempdir()) / 'fraud_demo_mlflow').as_uri())
    if MLFLOW_EXPERIMENT:
        mlflow.set_experiment(MLFLOW_EXPERIMENT)
    elif not os.environ.get('DATABRICKS_RUNTIME_VERSION'):
        mlflow.set_experiment('financial_fraud_teaching_demo')

    with mlflow.start_run(run_name='chronological_fraud_demo'):
        mlflow.log_params({
            'selected_model': selected_name, 'seed': SEED,
            'review_threshold': selected_threshold, 'validation_recall_target': TARGET_RECALL,
            'dataset_sha256': dataset_sha256, 'split_policy': 'chronological_60_20_20',
        })
        mlflow.log_metrics({f'test_{key}': float(value) for key, value in test_results.items()})
        signature = infer_signature(X_train.head(5), selected_model.predict(X_train.head(5)))
        mlflow.sklearn.log_model(
            selected_model, artifact_path='model', signature=signature,
            input_example=X_train.head(5),
        )
        mlflow.log_artifact(str(OUTPUT_DIR / 'run_manifest.json'))
        mlflow.log_artifact(str(OUTPUT_DIR / 'validation_metrics.csv'))
        mlflow.log_artifact(str(OUTPUT_DIR / 'test_metrics.csv'))
        print('MLflow run:', mlflow.active_run().info.run_id)
else:
    print('El registro opcional en MLflow está desactivado; el flujo principal ya terminó.')


## 16. Análisis del estudiante y extensiones controladas

Las respuestas se basan en las tablas y gráficas de mi propia corrida. Genie puede ayudar a entender, pero la idea es dominar los fundamentos.

1. Calcula la exactitud, el recall y el AP del dummy. ¿Por qué cuentan historias distintas?
2. Sigue un camino dentro del árbol de decisión. Explica Gini, profundidad, tamaño de hoja y pesos por clase.
3. ¿Qué modelo supervisado ganó en AP de validación? ¿El modelo con mayor ROC-AUC es necesariamente la mejor opción cuando el presupuesto de revisión es limitado?
4. Compara el umbral por defecto con el congelado: ¿cuántos falsos positivos y falsos negativos cambiaron en prueba? Explica por qué la meta de 80% de recall en validación no tiene por qué cumplirse en prueba.
5. Explica por qué el entrenamiento de Isolation Forest es no supervisado, pero nuestra evaluación con etiquetas no lo es. ¿Toda transacción inusual sería fraude?
6. Explica por qué rellenar todos los faltantes con cero, balancear los datos de prueba o ajustar el escalador antes de separar los datos puede llevar a conclusiones engañosas.
7. Discute el supuesto de quitar duplicados, las limitaciones del PCA que ya viene hecho, la falta de IDs de cliente y la privacidad. ¿Qué evidencia haría falta antes de desplegar algo así?
8. Reporta los intervalos de precisión/recall de la política congelada y la carga de alertas. ¿Un equipo de analistas podría con eso? ¿Por qué el escenario de costos inventado no es un caso de negocio?

**Cualquier extensión necesita su propio protocolo de desarrollo.** La ventana de prueba se queda cerrada. Usando solo entrenamiento y validación se pueden comparar profundidades 3/4/6, ajustar una política según el presupuesto de revisión o agregar gradient boosting.

**Opcional:** si se usa SMOTE o algún remuestreo, se aplica solo dentro de los folds de entrenamiento, nunca antes de separar. Para ajustar con datos temporales se usa validación forward-chaining, y para hacer una nueva afirmación final hace falta otra ventana futura de prueba. No se vale seguir ajustando después de ver los resultados de prueba de este notebook.

**Entregable sugerido:** un PDF con la auditoría, la tabla de cortes, la explicación del árbol, la comparación en validación, la justificación del umbral congelado, la matriz de confusión de prueba, la carga de trabajo, las limitaciones y las respuestas a las preguntas 1–8. Se incluyen el notebook y el manifiesto como archivos de reproducibilidad.


## Explicación de MLflow

MLflow sirve para registrar cada corrida de un experimento de forma ordenada: qué parámetros usé, qué métricas obtuve y qué archivos generé. Así puedo comparar corridas entre sí sin depender de la memoria o de capturas de pantalla. En esta sección registré la corrida `chronological_fraud_demo` en el workspace de Databricks.

**Parámetros registrados:** modelo seleccionado (`Random forest`), semilla (`42`), umbral de revisión elegido en validación (`0.109629`), meta de recall en validación (`0.80`), hash SHA-256 del dataset y la política de separación (`chronological_60_20_20`). Los parámetros dicen **con qué configuración** se entrenó y se decidió.

**Métricas registradas (prueba, con la política congelada):** AP, ROC-AUC, precisión, recall, F1, exactitud, tasa de alertas, y los conteos TN, FP, FN y TP. Las métricas dicen **qué tan bien funcionó** en datos que no se usaron para escoger nada.

**Artefactos registrados:** el modelo ajustado (con su firma de entrada), `run_manifest.json`, `validation_metrics.csv` y `test_metrics.csv`. Los artefactos permiten **reproducir** la corrida y revisar después qué se guardó exactamente.

Con MLflow puedo comparar, por ejemplo, otra profundidad del árbol o un umbral distinto: cada variante queda como un run con sus parámetros y métricas, y la comparación se hace sobre validación, no sobre prueba.

## Respuestas del estudiante: Oliver Camarena Palma (A00843123)

**Preguntas extra**

*¿Qué representa la variable objetivo y qué tan balanceadas están las clases?* `Class` vale 1 si la transacción es fraude y 0 si es legítima. Está muy desbalanceada: en el archivo original hay 492 fraudes en 284,807 transacciones (0.173%). Después de quitar duplicados exactos quedan 473 fraudes en 283,726 (0.167%).

*¿Por qué la exactitud sola puede engañar en detección de fraude?* El dummy que siempre dice "legítima" tiene una exactitud de 99.87% en prueba, pero su recall es 0: no detecta ningún fraude. Su AP es 0.0013, igual a la prevalencia de fraudes. La exactitud no distingue entre equivocarse en las transacciones legítimas (que son casi todas) y dejar pasar los fraudes, que es lo que realmente importa.

*¿Cómo afectan la precisión y el recall la interpretación de los resultados?* La precisión dice qué proporción de alertas son fraude de verdad; el recall dice qué proporción de fraudes encontramos. Se mueven en direcciones opuestas al cambiar el umbral. Con el umbral congelado en prueba, el recall es 0.838 pero la precisión es 0.380: encontramos casi todos los fraudes, pero de cada 100 alertas solo unas 38 son fraude.

*¿Qué modelo elegirías y qué evidencia lo respalda?* Según la regla declarada de antemano (mayor AP de validación entre los modelos supervisados), el ganador fue el Random forest, con AP 0.764 frente a 0.753 de la regresión logística, 0.508 del árbol y 0.018 de Isolation Forest. En prueba, con el umbral congelado (0.1096), detecta 62 de 74 fraudes y genera 101 falsos positivos. La diferencia con la regresión logística en AP es pequeña, así que no la tomaría como una victoria clara sin más evidencia.

*¿Qué limitaciones hay antes de aplicar esto a transacciones reales?* Los datos son de dos días de 2013, con variables PCA que ya vienen anonimizadas, sin IDs de cliente ni de tarjeta, y el benchmark no permite saber cómo se hizo el PCA. Tampoco hay información de contexto (comercio, ubicación, dispositivo). El umbral se escogió con muy pocos fraudes en validación (57), así que es inestable. Antes de usarlo haría falta una validación con datos recientes, monitoreo de deriva y revisar el impacto en clientes legítimos.

**Preguntas 1 a 8 (sección 16)**

1. *Dummy:* exactitud 99.87%, recall 0 y AP 0.0013 en prueba. La exactitud parece excelente, pero el recall en cero muestra que no sirve para detectar fraude. El AP igual a la prevalencia significa que no ordena nada: todas las transacciones tienen el mismo puntaje.

2. *Ruta del árbol:* el árbol de profundidad 4 empieza con `V14 <= -1.633`. Si esa condición se cumple, sigue con `V10 <= -1.880`, luego `V11 > 0.133` y por último `V14 <= -2.733`, y termina en una hoja de clase 1 (fraude). Cada decisión reduce la impureza de Gini, que mide qué tan mezcladas están las clases en un nodo (1 − p₀² − p₁²). La profundidad 4 limita a cuatro decisiones seguidas, y `min_samples_leaf=20` evita hojas con muy pocos casos. El `class_weight='balanced'` hace que cada fraude pese más en el entrenamiento, así que los puntajes de las hojas no son probabilidades reales.

3. *Modelo supervisado ganador:* el Random forest, por AP de validación. La regresión logística tiene mayor ROC-AUC en validación (0.969 frente a 0.958), pero el ROC-AUC no dice cuántas alertas genera. Con un presupuesto de revisión limitado importa el número de falsos positivos y la precisión en el punto de operación, no solo el orden global.

4. *Umbral por defecto (0.5) frente al congelado (0.1096) en prueba:* con 0.5 hay 56 fraudes detectados, 7 falsos positivos y 18 falsos negativos. Con el umbral congelado hay 62 detectados, 101 falsos positivos y 12 falsos negativos. Ganamos 6 fraudes y generamos 94 alertas más. La meta de 80% de recall en validación no tiene por qué cumplirse en datos nuevos, porque validación solo tenía 57 fraudes: en prueba se cumplió (0.838), pero el intervalo de Wilson al 95% va de 0.738 a 0.905, así que no es una garantía.

5. *Isolation Forest:* se entrena sin etiquetas, porque solo aprende qué transacciones son fáciles de aislar. Pero el umbral lo escogimos y lo evaluamos con etiquetas, así que esa decisión ya usa información de los fraudes. Su AP en validación es 0.018, muy bajo, y con su umbral por defecto la precisión es 0.018. No toda transacción inusual es fraude: una compra rara puede ser perfectamente legítima.

6. *Errores comunes:* rellenar faltantes con cero inventa datos que no existen y puede cambiar los resultados sin que lo notemos (el notebook prefiere fallar). Balancear el conjunto de prueba cambia la proporción real de fraudes y las métricas dejan de representar lo que pasaría en producción. Ajustar el escalador antes de separar los datos deja que información de validación o prueba influya en el entrenamiento (fuga de datos).

7. *Duplicados, PCA, privacidad y evidencia:* quitamos 1,081 duplicados exactos (0.38%) y mantuvimos la primera aparición, pero no sabemos si eran transacciones distintas indistinguibles. El PCA viene ya hecho y no sabemos si se construyó sin información del futuro. Sin IDs de cliente no podemos validar por cliente, y los datos anonimizados limitan la explicación. Antes de desplegar haría falta: datos propios y recientes, validación temporal con etiquetas reales, medición del impacto en clientes legítimos, revisión de privacidad y un plan de monitoreo.

8. *Intervalos y carga de trabajo (prueba, umbral congelado):* recall 0.838 (IC 95%: 0.738–0.905) y precisión 0.380 (IC 95%: 0.309–0.457). La política mandó a revisión 163 de 56,744 transacciones, unas 28.7 por cada 10,000. La ventana de prueba cubre aproximadamente las últimas 10 horas del periodo; si ese ritmo se mantuviera las 24 horas, serían unas 400 alertas diarias. Eso depende de la capacidad del equipo de analistas, que no conozco. El escenario de costos (fraude no detectado = 100, falsa alerta = 1) es inventado: no toma en cuenta el monto real de los fraudes, el costo de investigar, la recuperación ni el impacto en el cliente. Por eso el costo ilustrativo solo sirve para discutir el compromiso entre tipos de error, no como caso de negocio. Con esos supuestos, la política congelada cuesta 1,301, frente a 1,807 del umbral por defecto y 7,400 del dummy.

**Uso de Genie:** no utilicé Genie en esta actividad; todas las respuestas están basadas en las tablas y gráficas de mi corrida.


## 17. Referencias y atribución del dataset

1. Worldline y Machine Learning Group, Université Libre de Bruxelles.
   *Credit Card Fraud Detection* (página original del dataset):
   https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
   Antes de redistribuirlo, revisa los términos de uso/licencia vigentes del dataset original.
2. TensorFlow. *Classification on imbalanced data*. Descripción oficial y espejo público del CSV:
   https://www.tensorflow.org/tutorials/structured_data/imbalanced_data
   CSV: https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv
3. Lucas, Y., y Jurgovsky, J. (2020). *Credit card fraud detection using machine learning:
   A survey*. arXiv:2010.06479. https://arxiv.org/abs/2010.06479
   Contexto sobre desbalance, evaluación, comportamiento temporal y concept drift.
4. Niu, X., Wang, L., y Yang, X. (2019). *A Comparison Study of Credit Card Fraud Detection:
   Supervised versus Unsupervised*. arXiv:1904.10604. https://arxiv.org/abs/1904.10604
   Motivación para comparar enfoques supervisados y no supervisados; este notebook no reproduce su protocolo ni sus resultados.
5. scikit-learn. *Decision Trees*. https://scikit-learn.org/stable/modules/tree.html
6. scikit-learn. API de *IsolationForest* y referencias del algoritmo original.
   https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
7. scikit-learn. *Precision-Recall*.
   https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html
8. scikit-learn. *Tuning the decision threshold for class prediction*.
   https://scikit-learn.org/stable/modules/classification_threshold.html
9. Databricks. *Import and export Databricks notebooks*.
   https://docs.databricks.com/aws/en/notebooks/notebook-export-import
10. Ejemplo original de Databricks con PaySim, incluido como contexto:
    https://pages.databricks.com/rs/094-YMS-629/images/financial-fraud-detection-decision-tree.html

Fuentes y acceso al dataset revisados el 12 de noviembre de 2026. Todo el código de modelado de este notebook es una implementación nueva con fines didácticos; los puntajes publicados de otros benchmarks no se presentan como resultados propios.
